# 03 — Agent Loop (LangGraph ReAct SOC Triage Agent)

**Owner:** Marston Ward (AIE / Team Lead) · **Course:** AAI-510 · Group 8

This notebook defines the **concrete LangGraph `StateGraph`** ReAct agent: its
**State**, **nodes**, **tools**, and **edges**, plus the headline deliverable
`classify_and_ticket()`. All logic lives in `src/soc_agent/agent.py` so the
notebook stays a thin, readable driver.

## Architecture

```
START → scope_guard ─(out of scope)→ reject → END
                    └(in scope)────→ reason ⇄ act      (ReAct loop, ≤ 5 iters)
                                         └(no more tools)→ classify_and_ticket → END
```

- **scope_guard** — rejects out-of-scope / irrelevant user queries (graceful).
- **reason** — the LLM step. A real provider chooses tools; the `mock` provider
  follows a deterministic tool sequence so the loop still exercises every tool.
- **act** — executes tool calls, injecting host/indicator context from State, and
  folds normalized results back into State.
- **classify_and_ticket** — LLM-driven MITRE ATT&CK classification, then writes an
  incident row to `gold.incident` when `z_score > 2.5 AND confidence > 0.7`.

## How it consumes Sai's gold layer (and the signature drift)

The agent calls Sai's gold UC functions through `src/soc_agent/gold_tools.py`,
which **adapts to the ACTUALLY shipped signatures** (not the proposal contract):

| Function | Proposal said | Shipped reality (what we adapt to) |
|----------|---------------|-------------------------------------|
| `score_anomaly` | `(src_ip, window_days)` → dict | **TABLE-VALUED** `(p_host_ip, p_window_min INT)` — window in **MINUTES** |
| `classify_threat` | — | scalar Python UC, JSON→JSON `{tactic, technique_id, confidence}` |
| `get_exposed_assets` | — | TABLE-VALUED, no args → `(host_ip, risk_flag, assessed_at)` |

`top_anomaly()` bridges the shipped TVF back to a single `{z_score, ...}` dict the
agent reasons over. Everything runs in **MOCK_MODE** by default (gold results are
mocked) so the agent needs **no live Spark session and no creds**.


In [1]:
# --- bootstrap: make src/soc_agent importable + default to MOCK_MODE ---
import os, sys
from pathlib import Path

# Walk up to the repo root (the dir that contains src/soc_agent).
_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "src" / "soc_agent").exists():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate src/soc_agent from " + str(_here))

sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)  # so ./mlruns and relative paths are consistent

# Default to zero-creds mock mode unless the grader set real env vars.
os.environ.setdefault("SOC_MOCK_MODE", "1")
os.environ.setdefault("MLFLOW_TRACKING_URI", "file:./mlruns")

from soc_agent import config
SETTINGS = config.get_settings()
print("Repo root        :", REPO_ROOT)
print("MOCK_MODE        :", SETTINGS.mock_mode)
print("LLM provider     :", SETTINGS.llm_provider, "(effective:", SETTINGS.effective_llm_provider() + ")")
print("Live Databricks  :", SETTINGS.use_live_databricks())


Repo root        : /Users/marston.ward/Documents/GitHub/msaai-510-group8-soc-triage-agent
MOCK_MODE        : True
LLM provider     : databricks (effective: mock)
Live Databricks  : False


In [2]:
from soc_agent import agent, gold_tools, llm as llm_module
import json, pandas as pd

# The LLM is built from config — provider/model are never hardcoded.
llm = llm_module.get_llm(settings=SETTINGS)
print("LLM ->", llm.provider, "/", llm.model, "(is_mock:", llm.is_mock, ")")


LLM -> mock / databricks-meta-llama-3-1-70b-instruct (is_mock: True )


## Tools

Four LangChain tools wrap the gold functions and API clients. A real LLM picks
which to call; the executor injects `host_ip` / `indicator` from State so even
empty-arg (mock) tool calls resolve correctly.

In [3]:
tools = agent.build_tools(SETTINGS)
for t in tools:
    print(f"- {t.name}: {t.description}")

# Sample direct tool invocation (gold score_anomaly via the TVF adapter):
print("\nscore_anomaly(WORKSTATION6) ->", gold_tools.top_anomaly("WORKSTATION6"))


- score_anomaly: Compute the anomaly z-score for a host over a window (in MINUTES).
- check_ip_reputation: Look up IP/host reputation via VirusTotal (mocked by default).
- lookup_exposed_ports: Enumerate open ports / service banners via Shodan (mocked by default).
- get_cve_context: Look up CVEs (CVSS>=7) for a software/service keyword via NVD (keyless).

score_anomaly(WORKSTATION6) -> {'host_ip': 'WORKSTATION6', 'z_score': 4.51, 'baseline_mean': 8.0, 'baseline_std': 3.0, 'event_count': 22, 'rows': 1}


## The compiled graph

We build the `StateGraph` and render its structure.

In [4]:
compiled = agent.build_agent(llm, SETTINGS)
try:
    print(compiled.get_graph().draw_ascii())
except Exception as e:
    g = compiled.get_graph()
    print("nodes:", [n for n in g.nodes])
    print("edges:", [(e.source, e.target) for e in g.edges])


nodes: ['__start__', 'scope_guard', 'reject', 'reason', 'act', 'classify_and_ticket', '__end__']
edges: [('__start__', 'scope_guard'), ('act', 'reason'), ('reason', 'act'), ('reason', 'classify_and_ticket'), ('scope_guard', 'reason'), ('scope_guard', 'reject'), ('classify_and_ticket', '__end__'), ('reject', '__end__')]


## `classify_and_ticket()` — MITRE labeling + incident ticket

Given an enriched context, the LLM returns a strict JSON ticket
(`tactic, technique_id, confidence, priority 1–5, severity, summary`). Invalid
JSON is **routed to manual review**, never silently passed (proposal's
output-schema-enforcement control). When escalation criteria are met, a row is
written matching the **exact `gold.incident` schema**:

`incident_id, host_ip, user_id, tactic, technique_id, confidence, z_score,
severity, payload_json, created_at, resolved_at`

In [5]:
from soc_agent import api_clients
from soc_agent.mocks import SAMPLE_EVENTS

gold_tools.reset_incident_store()
context = {
    "host_ip": "FILESRV1",
    "event_payload": SAMPLE_EVENTS[2],
    "z_score": gold_tools.top_anomaly("FILESRV1")["z_score"],
    "reputation": api_clients.VirusTotalClient(SETTINGS).check_ip_reputation("FILESRV1"),
    "exposed_ports": api_clients.ShodanClient(SETTINGS).lookup_exposed_ports("FILESRV1"),
    "cve_context": api_clients.NVDClient(SETTINGS, mock_mode=False).get_cve_context("SMB", limit=2),
}

result = agent.classify_and_ticket(context, llm, SETTINGS)
print("decision :", result["decision"], "| written:", result["written"])
print("ticket   :", json.dumps(result["classification"], indent=2))
print("\nincident row (gold.incident schema):")
print(json.dumps(result["incident"], indent=2, default=str))


decision : escalated | written: True
ticket   : {
  "tactic": "Persistence",
  "technique_id": "T1543",
  "confidence": 0.88,
  "priority": 4,
  "severity": "HIGH",
  "summary": "[databricks-meta-llama-3-1-70b-instruct] Persistence (T1543) on FILESRV1 \u2014 z=3.10, VT malicious=7."
}

incident row (gold.incident schema):
{
  "incident_id": "25991ee8-055a-49bd-947e-69270ba853c4",
  "host_ip": "FILESRV1",
  "user_id": "administrator",
  "tactic": "Persistence",
  "technique_id": "T1543",
  "confidence": 0.88,
  "z_score": 3.1,
  "severity": "HIGH",
  "payload_json": "{\"EventID\": 4697, \"host_ip\": \"FILESRV1\", \"user_id\": \"administrator\", \"ProcessName\": \"services.exe\", \"CommandLine\": \"sc create evil binpath= C:\\\\temp\\\\x.exe\", \"event_type\": \"Security\", \"event_ts\": \"2026-05-31T05:02:51Z\"}",
  "created_at": "2026-06-02T17:34:44.870751+00:00",
  "resolved_at": null
}


## Full ReAct run on a sample SIEM event (in scope)

Watch the reasoning trace: scope check → tool calls → classify → ticket.

In [6]:
from soc_agent.mocks import SAMPLE_EVENTS
gold_tools.reset_incident_store()

run = agent.run_triage(
    "Investigate the encoded PowerShell execution alert on host WORKSTATION6.",
    SAMPLE_EVENTS[1],
    llm=llm, settings=SETTINGS,
)
print("DECISION:", run["decision"], "| iterations:", run["iterations"])
print("\nReasoning trace:")
for step in run["trace"]:
    print("  -", step)
print("\nFinal message:", run["final_message"])
print("\nIncidents written to gold.incident (mock store):", len(gold_tools.get_incident_store()))


DECISION: escalated | iterations: 4

Reasoning trace:
  - scope_guard: in_scope=True
  - reason[0]: tool_calls=['score_anomaly']
  - act: executed ['score_anomaly']
  - reason[1]: tool_calls=['check_ip_reputation']
  - act: executed ['check_ip_reputation']
  - reason[2]: tool_calls=['lookup_exposed_ports']
  - act: executed ['lookup_exposed_ports']
  - reason[3]: tool_calls=['get_cve_context']
  - act: executed ['get_cve_context']
  - reason[4]: final
  - classify_and_ticket: decision=escalated tactic=Execution written=True

Final message: Decision: ESCALATED. Execution (T1059), severity CRITICAL, confidence 0.82.

Incidents written to gold.incident (mock store): 1


## Out-of-scope rejection — **two explicit examples** (rubric requirement)

The `scope_guard` node rejects irrelevant queries gracefully, before any tool
call or LLM cost. Two worked examples:

In [7]:
for q in [
    "What's the weather in Paris this weekend?",            # example 1
    "Write me a poem about my cat and suggest a pasta recipe.",  # example 2
]:
    r = agent.run_triage(q, None, llm=llm, settings=SETTINGS)
    print(f"QUERY   : {q}")
    print(f"DECISION: {r['decision']}")
    print(f"RESPONSE: {r['final_message']}\n")


QUERY   : What's the weather in Paris this weekend?
DECISION: rejected
RESPONSE: This request is outside the SOC triage agent's scope. I only handle security alert triage: anomaly scoring, IP/host threat-intel enrichment, CVE context, and MITRE ATT&CK incident ticketing. Please ask about a host, alert, or security event.

QUERY   : Write me a poem about my cat and suggest a pasta recipe.
DECISION: rejected
RESPONSE: This request is outside the SOC triage agent's scope. I only handle security alert triage: anomaly scoring, IP/host threat-intel enrichment, CVE context, and MITRE ATT&CK incident ticketing. Please ask about a host, alert, or security event.



In [8]:
# Confirm the incident table view (mock store) matches the gold.incident schema.
store = gold_tools.get_incident_store()
print("gold.incident columns:", gold_tools.INCIDENT_COLUMNS)
pd.DataFrame(store)[gold_tools.INCIDENT_COLUMNS] if store else "no incidents yet"


gold.incident columns: ['incident_id', 'host_ip', 'user_id', 'tactic', 'technique_id', 'confidence', 'z_score', 'severity', 'payload_json', 'created_at', 'resolved_at']


,incident_id,host_ip,user_id,tactic,technique_id,confidence,z_score,severity,payload_json,created_at,resolved_at
0,59e1bafd-c577-4bd1-9e9e-d4eb030d5cbc,WORKSTATION6,jdoe,Execution,T1059,0.82,4.51,CRITICAL,"{""EventID"": 4688, ""host_ip"": ""WORKSTATION6"", ""...",2026-06-02T17:34:45.067625+00:00,None


### Summary

- A concrete `StateGraph` ReAct agent with explicit State, nodes, tools, edges.
- `classify_and_ticket()` produces schema-validated tickets and writes
  `gold.incident` rows; bad JSON → manual review.
- Out-of-scope queries are rejected gracefully (2 examples shown).
- Adapts to the **shipped** `score_anomaly` TVF (minutes), not the proposal dict.

Next: `05_evaluation.ipynb` traces these runs in MLflow and compares two
Databricks-served models on the same inputs.